In [1]:
import ot_utils
import numpy as np
import matplotlib.pyplot as plt

# import energyflow as ef
import itertools
from tqdm import tqdm

%matplotlib inline

dataset_length = 4000
num_dataset = 8
num_particles = 150
num_attr = 4

data_path = "../medium_dataset/"
file_names = [
    "QCD_4k.npz",
    "HToBB_4k.npz",
    "HToCC_4k.npz",
    "HToGG_4k.npz",
    "HToWW4Q_4k.npz",
    "ZToQQ_4k.npz",
    "WToQQ_4k.npz",
    "TTBar_4k.npz",
]

all_events = np.zeros((num_dataset * dataset_length, num_particles, num_attr))
all_labels = np.zeros(num_dataset * dataset_length)

event_blocks = np.zeros((num_dataset, dataset_length, num_particles, num_attr))

for i, file in enumerate(file_names):
    f = np.load(data_path + file)

    x = f["x"]
    y = f["y"]

    all_events[i * dataset_length : (i + 1) * dataset_length] = x
    all_labels[i * dataset_length : (i + 1) * dataset_length] = y

# all_distance_matrix = utils.calcOTDistance(all_events)
# eightk_distance_matrix = utils.parallel_OT_square(event_set=all_events, events_per_thread=50)
# np.save('8k_distance_matrix', eightk_distance_matrix)

In [2]:
def load_completed(path):
    """Read completed (i,j) pairs from progress file, return empty set if file missing."""
    try:
        with open(path, 'r') as f:
            return {
                tuple(map(int, line.split(',')))
                for line in f
                if line.strip()
            }
    except FileNotFoundError:
        return set()

def save_progress(path, i, j):
    """Append a completed (i,j) pair to progress file."""
    with open(path, 'a') as f:
        f.write(f'{i},{j}\n')

progress_file  = 'progress.txt'
completed = load_completed(progress_file)
total     = num_dataset * (num_dataset + 1) // 2
pairs     = itertools.combinations_with_replacement(range(num_dataset), 2)

with tqdm(total=total, desc='blocks') as pbar:
    for i, j in pairs:
        if (i, j) in completed:
            pbar.update()
            # print(i, j)
            continue

        # slice out your two event-blocks
        start_i = i * dataset_length
        end_i   = (i + 1) * dataset_length
        start_j = j * dataset_length
        end_j   = (j + 1) * dataset_length

        if i == j:
            block = ot_utils.parallel_OT_square(
                all_events[start_i:end_i],
                events_per_thread=20,
            )
        else:
            block = ot_utils.parallel_OT_non_square(
                all_events[start_i:end_i],
                all_events[start_j:end_j],
                events_per_thread=20,
            )

        # save and record progress
        np.save(f'w2_block_{i}_{j}.npy', block)
        save_progress(progress_file, i, j)
        pbar.update()

blocks: 100%|██████████| 36/36 [2:26:36<00:00, 244.34s/it]  


In [ ]:
n = 8
fig, axes = plt.subplots(n, n, figsize=(16, 16))

for i in range(n):
    for j in range(n):
        ax = axes[i, j]
        if j >= i:
            # load the block_i_j matrix
            mat = np.load(f"w2_block_{i}_{j}.npy")
            # extract non-zero entries
            data = mat[mat != 0]
            # plot histogram
            ax.hist(data, bins=50)
            # set x-axis limits
            ax.set_xlim(0, 0.5)
            ax.set_ylim(0, 110000)
            ax.set_title(f"w2_block_{i}_{j}", fontsize=8)
            # only label bottom row
            if i == n - 1:
                ax.set_xlabel("Distance", fontsize=7)
            else:
                ax.set_xticks([])
            # only label first column
            if j == 0:
                ax.set_ylabel("Count", fontsize=7)
            else:
                ax.set_yticks([])
        else:
            # hide the lower-triangle axes
            ax.axis("off")

fig.suptitle("Histograms of Non-zero Distances", y=1.02)
fig.tight_layout()
plt.show()


NameError: name 'plt' is not defined

In [ ]:
# self_matrix = eightk_distance_matrix[0000:1000, 7000:8000]
# # assume distance_matrix is your (N×N) symmetric NumPy array
# # get the indices of the upper-triangle (k=1 skips the diagonal)
# # i, j = np.triu_indices(self_matrix.shape[0], k=1)

# # pick out those entries
# # unique_dists = self_matrix[i, j]
# unique_dists = self_matrix

# mask = np.where(unique_dists != 0)

# unique_dists = unique_dists[mask]
# # now plot them
# plt.figure()
# plt.hist(unique_dists, bins=50)
# plt.xlabel("Distance")
# plt.ylabel("Count")
# plt.title("Histogram of Unique Pairwise Distances")
# plt.show()